# 139 · Doc Vision Agent

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/main/examples/139-doc-vision-agent/doc_vision_agent_workbook.ipynb)

**What this notebook teaches:** How to render PDF pages as PNG images with `pypdfium2`, encode them for the vision API, and prompt GPT-4o-mini to extract **structured JSON** from each page — one `extract_page` call per page, results collected by a two-node LangGraph workflow.

**Two halves:**
- **Part A (cells 1–4):** Pure Python — run anywhere, no API key needed. You'll see the PDF-to-PNG pipeline, inspect rendered image dimensions, and mock the JSON extraction result.
- **Part B (cells 5–7):** Requires `OPENAI_API_KEY`. Runs the full load → extract workflow against a public PDF.

| Concept | Why it matters |
|---------|----------------|
| `pypdfium2` → PIL → PNG | PDFs are vector; the vision API needs raster pixels |
| `scale=2` render | 2× scale gives ~150 DPI — readable text without huge file sizes |
| Schema prompt | Telling the model the exact JSON shape is faster and more reliable than free-form extraction |
| `strip("```json")` cleanup | LLMs often wrap JSON in markdown fences — strip before `json.loads()` |
| Fail-fast setup | Part B checks connectivity before iterating over every page |


In [ ]:
%pip install -q openai pypdfium2 httpx python-dotenv langgraph


## Part A · Cell 1 — The PDF Rendering Pipeline

PDFs are **page description programs**, not images. Before a vision model can read a page, you must **rasterize** it — convert the vector commands into a grid of pixels.

The pipeline in `src/tools.py`:

```
PDF bytes
    └─► pypdfium2.PdfDocument(bytes)   # parse the PDF in memory
            └─► doc[page_index]        # access one page
                    └─► page.render(scale=2)    # rasterize at 2× DPI
                            └─► bitmap.to_pil()     # convert to PIL Image
                                    └─► img.save(buf, format="PNG")  # encode as PNG
                                            └─► base64.b64encode(buf.getvalue())
```

**Why `scale=2`?**  
PDF coordinates are in 72-DPI points. `scale=2` gives 144 effective DPI — enough for the model to read body text and captions clearly. `scale=4` would give 288 DPI but quadruple the image size and token cost.

**Why PNG instead of JPEG?**  
PNG is lossless. Text documents have hard edges (characters, lines) that JPEG compression blurs, which can cause the model to misread digits or punctuation. Use PNG for text; JPEG is fine for photos.


In [ ]:
# Part A · Cell 2 — Render the first page of a public PDF to PNG
# No API key needed

import base64
import io
import httpx
import pypdfium2 as pdfium

# A small, freely available PDF (W3C WCAG 2.1 spec)
SAMPLE_PDF_URL = "https://www.w3.org/WAI/WCAG21/wcag21.pdf"

print(f"Fetching PDF from: {SAMPLE_PDF_URL}")
print("(This may take a moment for a large PDF...)")
resp = httpx.get(SAMPLE_PDF_URL, timeout=30)
resp.raise_for_status()
pdf_bytes = resp.content

# Open and inspect the document
doc = pdfium.PdfDocument(pdf_bytes)
print(f"\nPDF loaded: {len(doc)} pages, {len(pdf_bytes)/1024:.1f} KB")

# Render page 0 at scale=2
page = doc[0]
width_pt, height_pt = page.get_width(), page.get_height()
bitmap = page.render(scale=2)
pil_img = bitmap.to_pil()

print(f"\nPage 0 dimensions in PDF points : {width_pt:.0f} x {height_pt:.0f} pt")
print(f"Rendered image dimensions (2x)  : {pil_img.width} x {pil_img.height} px")
print(f"Image mode                       : {pil_img.mode}")

# Encode to PNG base64
buf = io.BytesIO()
pil_img.save(buf, format="PNG")
b64_page0 = base64.b64encode(buf.getvalue()).decode()

print(f"\nBase64 length of page 0 PNG: {len(b64_page0):,} chars")
print(f"First 60 chars: {b64_page0[:60]}...")


## Part A · Cell 3 — The Schema Prompt Approach

**Prompt engineering for structured output** works by telling the model exactly what JSON shape to produce and instructing it to respond *only* with JSON.

```python
schema_prompt = """
Extract the following fields from this document page as JSON:
{
  "title": "string or null",
  "date": "string or null",
  "key_numbers": ["list of numeric values mentioned"],
  "summary": "one sentence summary"
}
"""
```

The text message appended to the image block is:
```
<schema_prompt>\n\nRespond with valid JSON only.
```

**Why not use `response_format={"type": "json_object"}`?**  
That flag forces valid JSON but does *not* enforce the schema. With a clear schema in the prompt, the model follows the shape reliably without needing `instructor` or function-calling overhead.

**The fence-stripping pattern** handles common LLM output noise:
```python
text = resp.choices[0].message.content.strip()
text = text.strip("```json").strip("```")  # remove markdown fences if present
json.loads(text)
```


In [ ]:
# Part A · Cell 4 — Mock extract_page: show what the LLM response JSON looks like
# Simulates the output of src/tools.py extract_page() without an API call

import json

SCHEMA_PROMPT = """Extract the following fields from this document page as JSON:
{
  "title": "string or null",
  "date": "string or null",
  "key_numbers": ["list of numeric values mentioned"],
  "summary": "one sentence summary"
}"""


def mock_llm_response(page_content_hint: str) -> str:
    """Simulate the raw LLM response (sometimes with markdown fences)."""
    responses = {
        "cover": '{\n  "title": "Web Content Accessibility Guidelines (WCAG) 2.1",\n  "date": "5 June 2018",\n  "key_numbers": ["2.1", "1.0", "2.0"],\n  "summary": "This document defines WCAG 2.1, which extends WCAG 2.0 with new success criteria for mobile, low vision, and cognitive accessibility."\n}',
        "toc": '```json\n{\n  "title": "Table of Contents",\n  "date": null,\n  "key_numbers": ["1", "2", "3", "4"],\n  "summary": "The table of contents lists four principles: Perceivable, Operable, Understandable, and Robust."\n}\n```',
        "criteria": '  {"title": null, "date": null, "key_numbers": ["1.1.1", "1.4.3", "4.5:1"], "summary": "Success criterion 1.1.1 requires text alternatives for all non-text content."}  ',
    }
    return responses.get(page_content_hint, '{"title": null, "date": null, "key_numbers": [], "summary": "Page with no extractable structured content."}')


def mock_extract_page(b64_png: str, schema_prompt: str, page_hint: str = "cover") -> dict:
    """Mirrors src/tools.py extract_page() but uses a simulated response."""
    raw = mock_llm_response(page_hint)
    # Same fence-stripping logic as the real function
    text = raw.strip().strip("```json").strip("```").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"raw": text}


print("=" * 60)
print("Simulating extract_page() on 3 different page types:\n")

for hint in ["cover", "toc", "criteria"]:
    result = mock_extract_page(b64_page0, SCHEMA_PROMPT, page_hint=hint)
    result["page"] = ["cover", "toc", "criteria"].index(hint) + 1
    print(f"--- Page type: {hint} ---")
    print(json.dumps(result, indent=2))
    print()

print("Note: the fence-stripping handles all three response formats above correctly.")


---
## Part B · Live API Run

The cells below require `OPENAI_API_KEY` in your `.env` file.

The full workflow is a two-node LangGraph graph:
```
START → load_node → extract_node → END
          (fetch PDF,          (loop over pages,
           count pages)         call vision API per page)
```

**Warning:** The WCAG PDF has 60+ pages. Cell 7 only processes the first 3 pages to keep cost and latency low. Adjust `MAX_PAGES` to process more.


In [ ]:
# Part B · Cell 5 — Fail-fast setup
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY", "")
if not api_key:
    raise EnvironmentError(
        "OPENAI_API_KEY not set.\n"
        "Add it to your .env file or set the environment variable, then re-run this cell."
    )

client = OpenAI(api_key=api_key)

try:
    models = client.models.list()
    print("Connected to OpenAI API.")
except Exception as e:
    raise ConnectionError(f"Cannot reach OpenAI API: {e}") from e

print(f"PDF source : {SAMPLE_PDF_URL}")
print(f"PDF pages  : {len(doc)}")
print("Ready to run extraction workflow.")


In [ ]:
# Part B · Cell 6 — Run the LangGraph load → extract workflow
# Processes up to MAX_PAGES from the PDF to control cost

from langgraph.graph import StateGraph, END
from typing import TypedDict

MAX_PAGES = 3  # increase to process more pages


class DocState(TypedDict):
    pdf_source: str
    schema_prompt: str
    page_count: int
    results: list


def pdf_page_to_b64_live(pdf_bytes: bytes, page_index: int) -> str:
    doc_local = pdfium.PdfDocument(pdf_bytes)
    page = doc_local[page_index]
    bitmap = page.render(scale=2)
    img = bitmap.to_pil()
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode()


def extract_page_live(b64_png: str, schema_prompt: str, _client, model: str) -> dict:
    resp = _client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": [
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64_png}"}},
            {"type": "text", "text": schema_prompt + "\n\nRespond with valid JSON only."},
        ]}],
        max_tokens=1024,
    )
    text = resp.choices[0].message.content.strip().strip("```json").strip("```").strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"raw": text}


def load_node(state: DocState, config: dict) -> DocState:
    _pdf_bytes = httpx.get(state["pdf_source"], timeout=30).content
    _doc = pdfium.PdfDocument(_pdf_bytes)
    return {**state, "_pdf_bytes": _pdf_bytes, "page_count": min(MAX_PAGES, len(_doc))}


def extract_node(state: DocState, config: dict) -> DocState:
    _client = config["configurable"]["client"]
    model = config["configurable"].get("model", "gpt-4o-mini")
    _pdf_bytes = state["_pdf_bytes"]
    results = []
    for i in range(state["page_count"]):
        print(f"  Extracting page {i + 1}/{state['page_count']}...", end=" ")
        b64 = pdf_page_to_b64_live(_pdf_bytes, i)
        result = extract_page_live(b64, state["schema_prompt"], _client, model)
        result["page"] = i + 1
        results.append(result)
        print("done")
    return {**state, "results": results}


graph = StateGraph(DocState)
graph.add_node("load", load_node)
graph.add_node("extract", extract_node)
graph.set_entry_point("load")
graph.add_edge("load", "extract")
graph.add_edge("extract", END)
workflow = graph.compile()

cfg = {"configurable": {"client": client, "model": "gpt-4o-mini"}}

print(f"Running workflow on {SAMPLE_PDF_URL} (first {MAX_PAGES} pages)...\n")
output = workflow.invoke(
    {"pdf_source": SAMPLE_PDF_URL, "schema_prompt": SCHEMA_PROMPT, "page_count": 0, "results": []},
    config=cfg,
)
print(f"\nProcessed {output['page_count']} pages.")


In [ ]:
# Part B · Cell 7 — Display extracted results

print("=" * 60)
print("Extracted structured data per page:")
print("=" * 60)
for page_result in output["results"]:
    print(f"\nPage {page_result.get('page', '?')}:")
    print(json.dumps(page_result, indent=2))
